# Come RTR spiega una singola istanza

Questo notebook mostra, passo per passo, come viene spiegato il punteggio di un documento in RuleTreeRank. Il punteggio finale è la somma di due contributi:

    f(x) = r(x) + s(x)

- r(x) è il punteggio grezzo del primo stadio: un albero poco profondo manda il documento in una foglia e gli assegna la media delle etichette di quella foglia.
- s(x) è la correzione del secondo stadio: dentro la foglia si cercano i documenti più vicini (kNN) usando una distanza appresa, e si fa la media dei loro residui.

Alleniamo il modello con RuleCard come modello di distanza e spieghiamo una istanza reale: quale foglia, quali vicini, con quali regole, e come si arriva al punteggio finale. Alla fine confrontiamo i vicini scelti da RuleCard con quelli scelti dal PDT.

Usiamo di proposito RuleTreeRank base su dati sintetici, con foglie abbastanza grandi: così la distanza appresa viene davvero usata dal kNN e la spiegazione è significativa.

In [1]:
import sys
from pathlib import Path

REPO = Path.cwd()
while not (REPO / "ruletreerank").exists() and REPO != REPO.parent:
    REPO = REPO.parent
if str(REPO) not in sys.path:
    sys.path.insert(0, str(REPO))

import numpy as np
from RuleTree import RuleTreeRegressor
from ltr_utility import ModelParam
from ruletreerank import KNNRegFast, RuleTreeRank, RuleCardPairwiseDistance, PairwiseDistanceTree

# dati sintetici con una relazione lineare nota più un po' di rumore
rng = np.random.default_rng(0)
n, d = 300, 6
X = rng.normal(size=(n, d))
w = np.array([1.5, -0.8, 0.6, 0.0, 0.3, -0.4])
y = X @ w + rng.normal(0, 0.1, n)
q = np.repeat(np.arange(30), 10)  # 30 query da 10 documenti
cols = [f"f{j}" for j in range(d)]

def costruisci_rtr(distance_f):
    return RuleTreeRank(
        distance_f=distance_f,
        aggregation_f=ModelParam(KNNRegFast, {"n_neighbors": 5, "n_jobs": 1}),
        base_regressor=RuleTreeRegressor(max_depth=2, random_state=0),
        dist_objective="dist",
    )

rtr_rc = costruisci_rtr(ModelParam(RuleCardPairwiseDistance, {
    "base_regressor": ModelParam(RuleTreeRegressor, {"max_depth": 4, "random_state": 0}),
    "feature_diff": True, "feature_concat": True, "feature_sq_diff": False,
    "subsample": 1.0, "learning_rate": 0.2, "max_n_iter": 20, "patience": 3,
}))
rtr_rc.fit(X, y, q)
print("modello allenato")

C:\Users\manzo\anaconda3\envs\rtr\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


modello allenato


## Primo stadio: il punteggio grezzo r(x)

Scegliamo una istanza da spiegare e guardiamo in quale foglia finisce e che punteggio grezzo riceve.

In [2]:
i = 7  # istanza da spiegare
xi = X[i:i+1]
qi = q[i:i+1]

r = rtr_rc.predict(xi, q=qi, output="score")[0]
foglia = rtr_rc._shallow_dt.apply(xi)[0]

print("istanza:", np.round(xi[0], 3))
print("foglia raggiunta:", foglia, "  (la sigla codifica il percorso di decisione nell'albero)")
print(f"punteggio grezzo r(x) = {r:.4f}  (media delle etichette della foglia)")

istanza: [ 1.346  0.781  0.264 -0.314  1.458  1.96 ]
foglia raggiunta: Rrr   (la sigla codifica il percorso di decisione nell'albero)
punteggio grezzo r(x) = 2.4036  (media delle etichette della foglia)


## Secondo stadio: la correzione s(x)

Dentro la foglia si cercano i cinque documenti di training più vicini all'istanza, usando la distanza appresa da RuleCard. La correzione è la media dei loro residui (residuo = etichetta vera meno punteggio grezzo).

In [3]:
agg = rtr_rc._leaf_dist_map[foglia]
dist_model = agg.custom_metric_func
print("documenti di training nella foglia:", agg._fit_X.shape[0])

idx = agg.knn_neighbors_fast(xi).astype(int)[0]
residui_vicini = agg._y[idx].ravel()
distanze = dist_model.predict(np.repeat(xi, len(idx), axis=0), agg._fit_X[idx])

print("\nvicini scelti (indice nella foglia | distanza appresa | residuo):")
for k in range(len(idx)):
    print(f"  vicino {idx[k]:>3}   dist={distanze[k]:.4f}   residuo={residui_vicini[k]:+.4f}")

s = rtr_rc.predict(xi, q=qi, output="corr")[0]
print(f"\ncorrezione s(x) = {s:.4f}")
print(f"media dei residui dei vicini = {residui_vicini.mean():.4f}  (coincide con s per costruzione)")

documenti di training nella foglia: 42

vicini scelti (indice nella foglia | distanza appresa | residuo):
  vicino   1   dist=0.0000   residuo=-1.2071
  vicino  29   dist=6.1360   residuo=+1.3743
  vicino  14   dist=6.6242   residuo=-0.4979
  vicino   5   dist=6.6722   residuo=+0.1520
  vicino   7   dist=7.6311   residuo=+0.6407



correzione s(x) = 0.0924
media dei residui dei vicini = 0.0924  (coincide con s per costruzione)


## Perché quei vicini: le regole di RuleCard

La distanza tra due documenti è una somma di voci additive, una per ogni round di boosting. Ogni voce guarda la differenza assoluta di una feature tra i due documenti. Le prime voci sono le più importanti, perché a ogni round si sceglie la feature che riduce di più l'errore rimasto.

In [4]:
regole = dist_model.get_rules(columns_names=cols)
print(f"numero di voci additive: {len(regole)}  (fallback attivo: {dist_model._fallback})\n")
print("prime voci della scheda (ognuna è un alberello su una feature di differenza):")
for v in regole[:5]:
    print("  su", v["features"], "->", v["rules"])

numero di voci additive: 20  (fallback attivo: False)

prime voci della scheda (ognuna è un alberello su una feature di differenza):
  su ['diff(f2)'] -> 

{'node_id': 'R', 'is_leaf': False, 'prediction': 0.10106137993854176, 'prediction_probability': 6.478139399047021, 'log_odds': nan, 'prediction_classes_': array([-10.00921362,  -8.66098342,  -8.57052427,  -8.1741402 ,
        -7.980059  ,  -7.7563164 ,  -7.51068281,  -7.36134372,
        -7.35443362,  -7.31858377,  -7.30802101,  -7.08417034,
        -6.90006438,  -6.81543389,  -6.69721587,  -6.48900452,
        -6.42655281,  -6.2552954 ,  -6.17049087,  -5.85242044,
        -5.76924141,  -5.72554758,  -5.62744017,  -5.51270761,
        -5.33075152,  -5.3044095 ,  -5.06570286,  -4.95722831,
        -4.91251294,  -4.87651575,  -4.64276896,  -4.59949419,
        -4.3651666 ,  -4.32945933,  -4.06299289,  -4.03511921,
        -4.01804638,  -3.97524506,  -3.87198689,  -3.75678022,
        -3.67170875,  -3.62491889,  -3.60148674,  -3.56722284,
        -3.55975521,  -3.50676541,  -3.43926753,  -3.41343604,
        -3.40606226,  -3.33726355,  -3.17651222,  -3.01908088,
        -2.96143328,  -2.8

{'node_id': 'R', 'is_leaf': False, 'prediction': 0.49390642716783095, 'prediction_probability': 5.681920254765885, 'log_odds': nan, 'prediction_classes_': array([-8.06270567, -7.89179405, -7.46238958, -6.84068877, -6.50567547,
       -6.40589672, -6.33577045, -6.20314653, -6.15152165, -6.10622751,
       -5.90663872, -5.83159807, -5.7609554 , -5.7573041 , -5.73625914,
       -5.73044445, -5.17866337, -5.12458426, -5.10917135, -4.7986138 ,
       -4.74063782, -4.59649508, -4.44461989, -4.40974106, -4.26382273,
       -4.2271673 , -4.11639564, -4.08177328, -4.04991048, -4.03861463,
       -3.99006041, -3.96115465, -3.95367244, -3.9346575 , -3.76291075,
       -3.67900444, -3.49562929, -3.46881411, -3.42337293, -3.41199195,
       -3.35320969, -3.34294825, -3.24412671, -3.19317745, -3.18157236,
       -3.18067435, -3.17267402, -2.97451234, -2.83384474, -2.80638411,
       -2.78392103, -2.67242686, -2.65382411, -2.50432701, -2.4689801 ,
       -2.38272356, -2.16521934, -2.01947029, -1.8859

## Il punteggio finale

Sommando i due contributi si ottiene f(x), e ordinando i documenti della query per f(x) decrescente si ottiene la posizione dell'istanza nel ranking.

In [5]:
f = rtr_rc.predict(xi, q=qi, output="full")[0]
print(f"f(x) = r(x) + s(x) = {r:.4f} + ({s:.4f}) = {r + s:.4f}")
print(f"f(x) restituito dal modello = {f:.4f}")

# posizione nel ranking della sua query
mask_q = q == qi[0]
f_query = rtr_rc.predict(X[mask_q], q=q[mask_q], output="full")
ordine = np.argsort(-f_query)
pos = int(np.where(ordine == np.where(np.flatnonzero(mask_q) == i)[0][0])[0][0]) + 1
print(f"posizione dell'istanza nella sua query: {pos} su {mask_q.sum()}")

f(x) = r(x) + s(x) = 2.4036 + (0.0924) = 2.4960
f(x) restituito dal modello = 2.4960


posizione dell'istanza nella sua query: 1 su 10


## Confronto con il PDT

Alleniamo lo stesso RTR ma con il PDT come modello di distanza e guardiamo la stessa istanza. I vicini e le distanze cambiano, perché la distanza è appresa in modo diverso: il PDT usa un solo albero, RuleCard una somma di voci ordinate per importanza.

In [6]:
rtr_pdt = costruisci_rtr(ModelParam(PairwiseDistanceTree, {
    "base_regressor": ModelParam(RuleTreeRegressor, {"max_depth": 4, "random_state": 0}),
    "feature_diff": True, "feature_concat": True, "feature_sq_diff": False,
    "subsample": 1.0,
}))
rtr_pdt.fit(X, y, q)

agg_p = rtr_pdt._leaf_dist_map[rtr_pdt._shallow_dt.apply(xi)[0]]
idx_p = agg_p.knn_neighbors_fast(xi).astype(int)[0]
dist_p = agg_p.custom_metric_func.predict(np.repeat(xi, len(idx_p), axis=0), agg_p._fit_X[idx_p])

print("vicini scelti da RuleCard:", idx.tolist())
print("vicini scelti dal PDT    :", idx_p.tolist())
print("\ndistanze RuleCard:", np.round(distanze, 3).tolist())
print("distanze PDT     :", np.round(dist_p, 3).tolist())
# quanti valori distinti restituisce ciascuna distanza su tutte le coppie
# della foglia: un albero con l foglie puo' darne al piu' l, la somma additiva
# di RuleCard non ha questo limite
tutte_rc = dist_model.predict(np.repeat(xi, len(agg._fit_X), axis=0), agg._fit_X)
tutte_pdt = agg_p.custom_metric_func.predict(np.repeat(xi, len(agg_p._fit_X), axis=0), agg_p._fit_X)
print(f"\nvalori distinti su {len(tutte_rc)} coppie:  RuleCard {len(np.unique(tutte_rc))}"
      f"  |  PDT {len(np.unique(tutte_pdt))}")

print("\nStessa istanza, distanza appresa diversa: il PDT discretizza in pochi")
print("valori, RuleCard somma tante voci e distingue coppia per coppia.")

vicini scelti da RuleCard: [1, 29, 14, 5, 7]
vicini scelti dal PDT    : [1, 14, 26, 29, 7]

distanze RuleCard: [0.0, 6.136, 6.624, 6.672, 7.631]
distanze PDT     : [5.484, 5.484, 5.484, 5.484, 8.885]

valori distinti su 42 coppie:  RuleCard 42  |  PDT 7

Stessa istanza, distanza appresa diversa: il PDT discretizza in pochi
valori, RuleCard somma tante voci e distingue coppia per coppia.


## In sintesi

La spiegazione di una istanza si legge così: il primo stadio dice in quale gruppo (foglia) finisce e con quale punteggio di partenza; il secondo stadio mostra quali documenti simili l'hanno corretta, di quanto, e in base a quali differenze di feature. Con RuleCard la parte di distanza è una scheda a punti leggibile voce per voce, con il PDT è un percorso in un albero.